In [1]:
# Import libraries
import matplotlib.pyplot as plt
import koreanize_matplotlib

import numpy as np
import pandas as pd

from urllib.request import urlopen
import json


import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import folium


## 1. 전처리된 2025년 생활인구 데이터 불러오기 

In [2]:
SPOP_2025_ADM = pd.read_csv(f'data\SPOP_2025_ADM.csv')
SPOP_2025_ADM

<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\leeja\AppData\Local\Temp\ipykernel_42348\3003728435.py:1: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  SPOP_2025_ADM = pd.read_csv(f'data\SPOP_2025_ADM.csv')


,YMD,TT,H_DNG_CD,SPOP,M00,M10,M20,M30,M40,M50,...,F60,F70,ADMI_CD,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,HDAYS,WEEKEND,WORKDAY
0,20250101,0,11110515,14205.95,70.38,669.04,851.96,1021.66,1032.29,1131.88,...,1049.37,1214.13,11110515,서울특별시,종로구,청운효자동,서울특별시 종로구 청운효자동,1,0,0
1,20250101,0,11110530,15529.34,29.76,483.48,1191.52,1383.22,1179.69,1260.15,...,1115.81,1122.42,11110530,서울특별시,종로구,사직동,서울특별시 종로구 사직동,1,0,0
2,20250101,0,11110540,3296.03,6.22,114.46,299.31,228.36,238.77,261.97,...,286.65,323.15,11110540,서울특별시,종로구,삼청동,서울특별시 종로구 삼청동,1,0,0
3,20250101,0,11110550,11377.86,70.98,511.48,655.00,724.95,805.07,989.88,...,985.28,1033.39,11110550,서울특별시,종로구,부암동,서울특별시 종로구 부암동,1,0,0
4,20250101,0,11110560,19725.03,108.25,789.01,1130.13,1273.91,1310.67,1670.46,...,1859.78,1606.73,11110560,서울특별시,종로구,평창동,서울특별시 종로구 평창동,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3731755,20251231,19,11230536,41645.33,111.51,860.29,2556.25,3803.83,3484.63,3891.25,...,3375.74,2975.48,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1
3731756,20251231,20,11230536,41793.42,111.40,878.63,2663.29,3991.45,3503.55,3845.54,...,3308.61,2921.69,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1
3731757,20251231,21,11230536,39425.33,105.04,821.77,2491.18,3914.05,3263.19,3570.64,...,3006.68,2719.46,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1
3731758,20251231,22,11230536,40861.42,103.68,847.92,2661.42,4094.05,3458.72,3639.98,...,3067.54,2867.92,11230536,서울특별시,동대문구,용신동,서울특별시 동대문구 용신동,0,0,1


## 2. 시간 변수 생성

In [3]:
# TT를 2자리 문자열로 변환 (0 -> '00')
SPOP_2025_ADM['TT_str'] = SPOP_2025_ADM['TT'].apply(lambda x: str(x).zfill(2))

# YMD(정수) + TT_str을 합쳐 datetime으로 변환
SPOP_2025_ADM['DATETIME'] = pd.to_datetime(
    SPOP_2025_ADM['YMD'].astype(str) + SPOP_2025_ADM['TT_str'],
    format='%Y%m%d%H'
)

SPOP_2025_ADM[['YMD', 'TT', 'DATETIME']]

,YMD,TT,DATETIME
0,20250101,0,2025-01-01 00:00:00
1,20250101,0,2025-01-01 00:00:00
2,20250101,0,2025-01-01 00:00:00
3,20250101,0,2025-01-01 00:00:00
4,20250101,0,2025-01-01 00:00:00
...,...,...,...
3731755,20251231,19,2025-12-31 19:00:00
3731756,20251231,20,2025-12-31 20:00:00
3731757,20251231,21,2025-12-31 21:00:00
3731758,20251231,22,2025-12-31 22:00:00


In [4]:
SPOP_2025_ADM.dtypes

YMD                  int64
TT                   int64
H_DNG_CD             int64
SPOP               float64
M00                float64
M10                float64
M20                float64
M30                float64
M40                float64
M50                float64
M60                float64
M70                float64
F00                float64
F10                float64
F20                float64
F30                float64
F40                float64
F50                float64
F60                float64
F70                float64
ADMI_CD              int64
SIDO_NM                str
SGG_NM                 str
ADMI_NM                str
FULL_NM                str
HDAYS                int64
WEEKEND              int64
WORKDAY              int64
TT_str                 str
DATETIME    datetime64[us]
dtype: object

## 3. 시간 단위 생활인구 시계열 변화 시각화

In [5]:
SPOP_2025_ADM.ADMI_NM.unique()

<StringArray>
[      '청운효자동',         '사직동',         '삼청동',         '부암동',         '평창동',
         '무악동',         '교남동',         '가회동', '종로1.2.3.4가동',     '종로5.6가동',
 ...
        '천호1동',        '천호2동',        '천호3동',        '성내1동',        '성내2동',
        '성내3동',          '길동',        '둔촌1동',        '둔촌2동',         '용신동']
Length: 425, dtype: str

In [6]:
SPOP_2025_ADM_ys1 = SPOP_2025_ADM.query("ADMI_NM == '역삼1동'")
SPOP_2025_ADM_ys1

,YMD,TT,H_DNG_CD,SPOP,M00,M10,M20,M30,M40,M50,...,ADMI_CD,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,HDAYS,WEEKEND,WORKDAY,TT_str,DATETIME
367,20250101,0,11680640,51132.72,60.59,1404.01,5528.64,7004.62,5157.58,3531.81,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,1,0,0,00,2025-01-01 00:00:00
792,20250101,1,11680640,49048.82,60.59,1319.43,5303.16,6886.27,4945.06,3418.32,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,1,0,0,01,2025-01-01 01:00:00
1217,20250101,2,11680640,47394.94,60.59,1270.97,5116.96,6633.87,4792.31,3268.63,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,1,0,0,02,2025-01-01 02:00:00
1642,20250101,3,11680640,46275.69,60.59,1162.78,4978.18,6552.17,4621.81,3130.82,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,1,0,0,03,2025-01-01 03:00:00
2067,20250101,4,11680640,45473.20,60.59,1086.04,4989.67,6403.59,4510.12,3094.78,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,1,0,0,04,2025-01-01 04:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3721242,20251231,19,11680640,103531.76,105.84,2085.67,11345.74,14305.45,10657.47,7462.88,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,0,0,1,19,2025-12-31 19:00:00
3721667,20251231,20,11680640,89096.71,112.77,2030.79,10185.18,11875.08,8910.91,6265.80,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,0,0,1,20,2025-12-31 20:00:00
3722092,20251231,21,11680640,77791.04,116.30,1950.38,9163.30,10369.17,7730.05,5525.70,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,0,0,1,21,2025-12-31 21:00:00
3722517,20251231,22,11680640,68800.06,111.92,1849.64,8095.05,9352.75,6791.07,4777.27,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,0,0,1,22,2025-12-31 22:00:00


In [7]:
fig = px.line(SPOP_2025_ADM_ys1, 
              x = 'DATETIME', 
              y = 'SPOP', 
              labels = {'DATETIME' : '날짜/시간', 'SPOP' : '생활인구'},
              title = '역삼1동 생활인구',
              template="simple_white"
             )

fig.update_yaxes(showgrid=True)
fig.update_xaxes(showgrid=True, tickangle=-90)
fig.update_traces(line_color='orange')
fig.update_xaxes(rangeslider_visible=True) # x축 범위를 설정

fig.show()
fig.write_html("html/1_1.html")


In [8]:
# 중심지 추출.
dongs = ['역삼1동', '종로1.2.3.4가동', '영등포동', '서교동']
SPOP_2025_ADM_ct = SPOP_2025_ADM[SPOP_2025_ADM['ADMI_NM'].isin(dongs)]
SPOP_2025_ADM_ct

,YMD,TT,H_DNG_CD,SPOP,M00,M10,M20,M30,M40,M50,...,ADMI_CD,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,HDAYS,WEEKEND,WORKDAY,TT_str,DATETIME
8,20250101,0,11110615,39242.07,55.43,1241.80,6785.55,5947.45,3225.98,3230.27,...,11110615,서울특별시,종로구,종로1.2.3.4가동,서울특별시 종로구 종로1.2.3.4가동,1,0,0,00,2025-01-01 00:00:00
213,20250101,0,11440660,68151.53,33.59,5717.09,14261.81,6489.11,3188.39,2540.23,...,11440660,서울특별시,마포구,서교동,서울특별시 마포구 서교동,1,0,0,00,2025-01-01 00:00:00
286,20250101,0,11560535,35712.10,48.94,671.56,4279.70,5491.11,2804.32,2539.45,...,11560535,서울특별시,영등포구,영등포동,서울특별시 영등포구 영등포동,1,0,0,00,2025-01-01 00:00:00
367,20250101,0,11680640,51132.72,60.59,1404.01,5528.64,7004.62,5157.58,3531.81,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,1,0,0,00,2025-01-01 00:00:00
433,20250101,1,11110615,32617.44,50.03,957.43,5427.53,5154.94,2796.28,2709.85,...,11110615,서울특별시,종로구,종로1.2.3.4가동,서울특별시 종로구 종로1.2.3.4가동,1,0,0,01,2025-01-01 01:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3722517,20251231,22,11680640,68800.06,111.92,1849.64,8095.05,9352.75,6791.07,4777.27,...,11680640,서울특별시,강남구,역삼1동,서울특별시 강남구 역삼1동,0,0,1,22,2025-12-31 22:00:00
3722583,20251231,23,11110615,56025.06,120.78,2003.36,8702.59,7574.55,4758.51,4234.12,...,11110615,서울특별시,종로구,종로1.2.3.4가동,서울특별시 종로구 종로1.2.3.4가동,0,0,1,23,2025-12-31 23:00:00
3722788,20251231,23,11440660,75863.29,30.65,6671.70,13827.08,7392.06,3680.72,3116.69,...,11440660,서울특별시,마포구,서교동,서울특별시 마포구 서교동,0,0,1,23,2025-12-31 23:00:00
3722861,20251231,23,11560535,41698.89,70.49,793.50,4645.67,6569.30,3314.34,2884.28,...,11560535,서울특별시,영등포구,영등포동,서울특별시 영등포구 영등포동,0,0,1,23,2025-12-31 23:00:00


In [9]:
fig = px.line(SPOP_2025_ADM_ct, 
              x = 'DATETIME', 
              y = 'SPOP', 
              labels = {'DATETIME' : '날짜/시간', 'SPOP' : '생활인구'},
              title = '주요 지역 생활인구',
              color='ADMI_NM',
              # markers = True,
              template="simple_white"
             )

fig.update_yaxes(showgrid=True)
fig.update_xaxes(showgrid=True, tickangle=-90)
fig.update_xaxes(rangeslider_visible=True) # x축 범위를 설정

fig.show()
fig.write_html("html/1_2.html")



## 4. 읍면동 / 시간대별 연평균(연앙) 생활인구 산출 및 시각화

In [10]:
SPOP_2025_ADM_mid = SPOP_2025_ADM.groupby(['SIDO_NM','SGG_NM','ADMI_NM','FULL_NM','ADMI_CD','TT_str'])['SPOP'].agg(['mean']).reset_index().rename(columns = {'mean' : 'SPOP_mid'})
SPOP_2025_ADM_mid

,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,ADMI_CD,TT_str,SPOP_mid
0,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,00,29353.457973
1,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,01,29511.448630
2,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,02,29689.065808
3,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,03,29763.615671
4,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,04,29782.422274
...,...,...,...,...,...,...,...
10219,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,19,22849.486301
10220,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,20,23391.094849
10221,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,21,23805.087014
10222,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,22,24206.777205


In [11]:
# csv 파일로 저장
SPOP_2025_ADM_mid.to_csv('data/SPOP_2025_ADM_mid.csv', index=False, encoding='utf-8-sig') 

In [12]:
# 다시 불러오기
SPOP_2025_ADM_mid = pd.read_csv(f'data\SPOP_2025_ADM_mid.csv')
SPOP_2025_ADM_mid

<>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\leeja\AppData\Local\Temp\ipykernel_42348\1972086617.py:2: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  SPOP_2025_ADM_mid = pd.read_csv(f'data\SPOP_2025_ADM_mid.csv')


,SIDO_NM,SGG_NM,ADMI_NM,FULL_NM,ADMI_CD,TT_str,SPOP_mid
0,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,0,29353.457973
1,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,1,29511.448630
2,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,2,29689.065808
3,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,3,29763.615671
4,서울특별시,강남구,개포1동,서울특별시 강남구 개포1동,11680660,4,29782.422274
...,...,...,...,...,...,...,...
10219,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,19,22849.486301
10220,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,20,23391.094849
10221,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,21,23805.087014
10222,서울특별시,중랑구,중화2동,서울특별시 중랑구 중화2동,11260610,22,24206.777205


In [13]:
# 3도심 추출.
dongs = ['역삼1동', '종로1.2.3.4가동', '영등포동', '서교동']

fig = px.line(SPOP_2025_ADM_mid[SPOP_2025_ADM_mid['ADMI_NM'].isin(dongs)], 
              x = 'TT_str', 
              y = 'SPOP_mid', 
              labels = {'TT_str' : '시각', 'SPOP_mid' : '연평균 생활인구'},
              title = '주요 지역 시간대별 연앙 생활인구',
              color='ADMI_NM',
              markers = True,   # 마커추가
              template="simple_white"
             )

fig.update_yaxes(showgrid=True)
fig.update_xaxes(showgrid=True, tickangle=-90)
# fig.update_xaxes(rangeslider_visible=True) # x축 범위를 설정

fig.show()
fig.write_html("html/1_3.html")


In [14]:
# 강남구

fig = px.line(SPOP_2025_ADM_mid[SPOP_2025_ADM_mid['SGG_NM']=='강남구'], 
              x = 'TT_str', 
              y = 'SPOP_mid', 
              labels = {'TT_str' : '시각', 'SPOP_mid' : '연평균 생활인구'},
              title = '주요 지역 시간대별 연앙 생활인구',
              color='ADMI_NM',
              markers = True,   # 마커추가
              template="simple_white"
             )

fig.update_yaxes(showgrid=True)
fig.update_xaxes(showgrid=True, tickangle=-90)
# fig.update_xaxes(rangeslider_visible=True) # x축 범위를 설정

fig.show()
fig.write_html("html/1_4.html")

In [15]:
# 강남구

fig = px.line(SPOP_2025_ADM_mid[SPOP_2025_ADM_mid['SGG_NM']=='강남구'], 
              x = 'TT_str', 
              y = 'SPOP_mid', 
              labels = {'TT_str' : '시각', 'SPOP_mid' : '연평균 생활인구'},
              title = '강남구 읍면동 시간대별 연앙 생활인구',
              facet_col = 'ADMI_NM',
              facet_col_wrap = 5,
              facet_row_spacing=0.09, # default is 0.07 when facet_col_wrap is used
              facet_col_spacing=0.02,
              # markers = True,   # 마커추가
              height = 800,
              template="simple_white"
             )

fig.update_yaxes(showgrid=True)
fig.update_xaxes(showgrid=True, tickangle=-90)
# fig.update_xaxes(rangeslider_visible=True) # x축 범위를 설정
fig.update_traces(line_color='orange')

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))


fig.show()
fig.write_html("html/1_5.html")


## 5. 행정동 시간대별 생활인구 패턴 유형화 (K-means 군집분석)

- 행정동(426개) 각각의 0~23시 평균 생활인구 곡선을 z-score로 표준화(패턴의 "모양"만 비교, 규모 효과 제거)
- elbow(관성) + silhouette score로 군집 수 k 결정
- K-means로 유형화 후 군집별 평균 패턴과 소속 행정동 확인

In [16]:
# 행정동 x 시간대 평균 생활인구 (연앙 패턴)
SPOP_2025_ADM_mid = (
    SPOP_2025_ADM
    .groupby(['SIDO_NM', 'SGG_NM', 'ADMI_NM', 'H_DNG_CD', 'TT'])['SPOP']
    .mean()
    .reset_index()
    .rename(columns={'SPOP': 'SPOP_mid'})
)
SPOP_2025_ADM_mid

,SIDO_NM,SGG_NM,ADMI_NM,H_DNG_CD,TT,SPOP_mid
0,서울특별시,강남구,개포1동,11680660,0,29353.457973
1,서울특별시,강남구,개포1동,11680660,1,29511.448630
2,서울특별시,강남구,개포1동,11680660,2,29689.065808
3,서울특별시,강남구,개포1동,11680660,3,29763.615671
4,서울특별시,강남구,개포1동,11680660,4,29782.422274
...,...,...,...,...,...,...
10219,서울특별시,중랑구,중화2동,11260610,19,22849.486301
10220,서울특별시,중랑구,중화2동,11260610,20,23391.094849
10221,서울특별시,중랑구,중화2동,11260610,21,23805.087014
10222,서울특별시,중랑구,중화2동,11260610,22,24206.777205


In [17]:
# 행정동을 행(row), 0~23시를 열(column)로 하는 24차원 패턴 행렬 생성
pattern_wide = SPOP_2025_ADM_mid.pivot(index='H_DNG_CD', columns='TT', values='SPOP_mid')
pattern_wide.columns = [f'H{h:02d}' for h in pattern_wide.columns]

# 행정동별로 자기 자신 기준 z-score 표준화 (규모 차이를 없애고 하루 패턴의 "모양"만 비교)
pattern_scaled = pattern_wide.sub(pattern_wide.mean(axis=1), axis=0).div(pattern_wide.std(axis=1), axis=0)

pattern_scaled

,H00,H01,H02,H03,H04,H05,H06,H07,H08,H09,...,H14,H15,H16,H17,H18,H19,H20,H21,H22,H23
H_DNG_CD,,,,,,,,,,,,,,,,,,,,,
11110515,-0.971817,-0.973649,-0.975742,-0.972184,-0.966937,-0.953769,-0.882291,-0.677393,-0.249287,0.222660,...,1.629218,1.636085,1.351846,0.834184,0.289908,-0.127436,-0.427061,-0.678487,-0.850667,-0.932381
11110530,-1.101141,-1.116341,-1.132532,-1.139827,-1.138812,-1.106982,-1.003738,-0.787730,-0.282377,0.451005,...,1.384874,1.352215,1.272378,1.118456,0.800128,0.251786,-0.110617,-0.394720,-0.687119,-0.904909
11110540,-0.979125,-0.982937,-0.984964,-0.983099,-0.979072,-0.947017,-0.876539,-0.687123,-0.418157,-0.007260,...,1.606418,1.635696,1.518050,1.115376,0.618393,0.045087,-0.353660,-0.632727,-0.809609,-0.902083
11110550,-0.762562,-0.756618,-0.824150,-0.846464,-0.840583,-0.825292,-0.681122,-0.565170,-0.261527,-0.003115,...,1.857308,1.671698,1.257766,0.625419,-0.094563,-0.607904,-0.850669,-0.929172,-0.804278,-0.650814
11110560,0.621595,0.827579,0.953413,0.994414,1.010034,1.038536,0.916439,0.906328,1.058845,0.700217,...,-0.464042,-0.815072,-1.217728,-1.582853,-1.883000,-1.710521,-1.333064,-0.879559,-0.396524,-0.039913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11740650,0.743255,0.688499,0.696950,0.721821,0.733093,0.719413,0.451918,0.106127,-0.719251,-1.440969,...,-1.310249,-1.128192,-0.892199,-0.343578,0.455178,1.161578,1.318845,1.193647,1.039252,0.916543
11740660,-0.880637,-0.976302,-1.052916,-1.114736,-1.150738,-1.159348,-1.196257,-1.279436,-1.112745,-0.394404,...,1.059349,0.975805,0.969103,1.091727,1.196694,1.147317,0.733266,0.229112,-0.225073,-0.491215
11740685,0.130499,-0.191929,-0.550016,-0.843368,-1.120385,-1.233037,-1.264633,-1.231761,-1.223464,-1.307890,...,0.207852,-0.058196,-0.175563,-0.392232,-0.398509,1.028821,1.536091,1.617277,1.490240,1.304440


In [18]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_range = range(2, 11)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(pattern_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(pattern_scaled, labels))

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=list(K_range), y=inertias, name='Inertia (elbow)', mode='lines+markers'), secondary_y=False)
fig.add_trace(go.Scatter(x=list(K_range), y=silhouettes, name='Silhouette score', mode='lines+markers'), secondary_y=True)
fig.update_layout(title='k별 elbow(관성) vs silhouette score', xaxis_title='k (군집 수)', template='simple_white')
fig.update_yaxes(title_text='Inertia', secondary_y=False)
fig.update_yaxes(title_text='Silhouette score', secondary_y=True)
fig.show()
fig.write_html("html/1_6.html")


In [19]:
# 위 elbow/silhouette 그래프를 보고 K 값을 조정하세요 (우선 5로 설정)
K = 3

km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
pattern_wide['CLUSTER'] = km_final.fit_predict(pattern_scaled)
pattern_scaled['CLUSTER'] = pattern_wide['CLUSTER'].values

# 군집별 시간대 패턴을 보고 부여한 유형 이름
CLUSTER_LABELS = {
    0: '주간활성화지역',
    1: '주거중심지역',
    2: '야간활성화지역',
}
pattern_wide['CLUSTER_NM'] = pattern_wide['CLUSTER'].map(CLUSTER_LABELS)
pattern_scaled['CLUSTER_NM'] = pattern_wide['CLUSTER_NM'].values

pattern_wide['CLUSTER_NM'].value_counts()

CLUSTER_NM
주거중심지역     239
주간활성화지역    138
야간활성화지역     49
Name: count, dtype: int64

In [20]:
# 군집별 평균 시간대 패턴 (표준화된 값) 시각화 -> 유형 해석용
hour_cols = [c for c in pattern_wide.columns if c.startswith('H')]

CLUSTER_COLORS = {
    '주거중심지역': 'orange',
    '주간활성화지역': 'red',
    '야간활성화지역': 'purple',
}

cluster_profile = pattern_scaled.groupby('CLUSTER_NM')[hour_cols].mean().reset_index()
cluster_profile_long = cluster_profile.melt(id_vars='CLUSTER_NM', var_name='HOUR', value_name='Z_SPOP')
cluster_profile_long['HOUR'] = cluster_profile_long['HOUR'].str.replace('H', '').astype(int)

fig = px.line(
    cluster_profile_long,
    x='HOUR', y='Z_SPOP', color='CLUSTER_NM',
    color_discrete_map=CLUSTER_COLORS,
    markers=True,
    labels={'HOUR': 'Hour', 'Z_SPOP': '표준화된 생활인구', 'CLUSTER_NM': '유형'},
    title=f'유형별 평균 시간대 생활인구 패턴 (K={K})',
    template='simple_white',
)
fig.update_xaxes(dtick=1)
fig.show()
fig.write_html("html/1_7.html")


In [21]:
# 행정동 이름과 군집 라벨 매핑
dong_cluster = (
    SPOP_2025_ADM_mid[['SIDO_NM', 'SGG_NM', 'ADMI_NM', 'H_DNG_CD']]
    .drop_duplicates()
    .merge(pattern_wide[['CLUSTER', 'CLUSTER_NM']].reset_index(), on='H_DNG_CD', how='left')
    .sort_values(['CLUSTER', 'SGG_NM', 'ADMI_NM'])
    .reset_index(drop=True)
)
dong_cluster

,SIDO_NM,SGG_NM,ADMI_NM,H_DNG_CD,CLUSTER,CLUSTER_NM
0,서울특별시,강남구,개포3동,11680675,0,주간활성화지역
1,서울특별시,강남구,논현1동,11680521,0,주간활성화지역
2,서울특별시,강남구,논현2동,11680531,0,주간활성화지역
3,서울특별시,강남구,대치1동,11680600,0,주간활성화지역
4,서울특별시,강남구,대치2동,11680610,0,주간활성화지역
...,...,...,...,...,...,...
421,서울특별시,은평구,대조동,11380570,2,야간활성화지역
422,서울특별시,은평구,응암3동,11380600,2,야간활성화지역
423,서울특별시,종로구,혜화동,11110650,2,야간활성화지역
424,서울특별시,중랑구,면목3.8동,11260575,2,야간활성화지역


In [22]:
len(SPOP_2025_ADM_mid.H_DNG_CD.unique())

426

## 6. 군집 결과 지도 시각화

- 2025-01-01 기준 행정동 경계 (출처: [vuski/admdongkor](https://github.com/vuski/admdongkor) 저장소, `ver20250101`) — 서울 426개 동, 우리 데이터의 1~7월 기준과 일치
- 주의: 이 파일의 `adm_cd`(8자리)는 우리 `H_DNG_CD`와 다른 코드체계. `adm_cd2`(10자리)의 앞 8자리가 실제로 일치하는 코드

In [23]:
import geopandas as gpd

gdf_admdong = gpd.read_file('data/HangJeongDong_ver20250101.geojson')

# 서울만 추출, adm_cd2 앞 8자리를 H_DNG_CD로 사용 (adm_cd는 다른 코드체계라 사용하면 안 됨)
gdf_seoul = gdf_admdong[gdf_admdong['sidonm'] == '서울특별시'].copy()
gdf_seoul['H_DNG_CD'] = gdf_seoul['adm_cd2'].str[:8].astype(int)

print(len(gdf_seoul), '서울 행정동 경계 로드')
gdf_seoul.head()

426 서울 행정동 경계 로드


,adm_nm,adm_cd2,sgg,sido,sidonm,sggnm,adm_cd,geometry,H_DNG_CD
0,서울특별시 종로구 사직동,1111053000,11110,11,서울특별시,종로구,11010530,"MULTIPOLYGON (((126.97689 37.57565, 126.97703 ...",11110530
1,서울특별시 종로구 삼청동,1111054000,11110,11,서울특별시,종로구,11010540,"MULTIPOLYGON (((126.98269 37.59507, 126.98337 ...",11110540
2,서울특별시 종로구 부암동,1111055000,11110,11,서울특별시,종로구,11010550,"MULTIPOLYGON (((126.97585 37.59656, 126.97359 ...",11110550
3,서울특별시 종로구 평창동,1111056000,11110,11,서울특별시,종로구,11010560,"MULTIPOLYGON (((126.97507 37.63139, 126.97649 ...",11110560
4,서울특별시 종로구 무악동,1111057000,11110,11,서울특별시,종로구,11010570,"MULTIPOLYGON (((126.96067 37.5808, 126.96281 3...",11110570


In [24]:
gdf_cluster = gdf_seoul.merge(dong_cluster, on='H_DNG_CD', how='left')
gdf_cluster

,adm_nm,adm_cd2,sgg,sido,sidonm,sggnm,adm_cd,geometry,H_DNG_CD,SIDO_NM,SGG_NM,ADMI_NM,CLUSTER,CLUSTER_NM
0,서울특별시 종로구 사직동,1111053000,11110,11,서울특별시,종로구,11010530,"MULTIPOLYGON (((126.97689 37.57565, 126.97703 ...",11110530,서울특별시,종로구,사직동,0,주간활성화지역
1,서울특별시 종로구 삼청동,1111054000,11110,11,서울특별시,종로구,11010540,"MULTIPOLYGON (((126.98269 37.59507, 126.98337 ...",11110540,서울특별시,종로구,삼청동,0,주간활성화지역
2,서울특별시 종로구 부암동,1111055000,11110,11,서울특별시,종로구,11010550,"MULTIPOLYGON (((126.97585 37.59656, 126.97359 ...",11110550,서울특별시,종로구,부암동,0,주간활성화지역
3,서울특별시 종로구 평창동,1111056000,11110,11,서울특별시,종로구,11010560,"MULTIPOLYGON (((126.97507 37.63139, 126.97649 ...",11110560,서울특별시,종로구,평창동,1,주거중심지역
4,서울특별시 종로구 무악동,1111057000,11110,11,서울특별시,종로구,11010570,"MULTIPOLYGON (((126.96067 37.5808, 126.96281 3...",11110570,서울특별시,종로구,무악동,0,주간활성화지역
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
421,서울특별시 강동구 암사1동,1174057000,11740,11,서울특별시,강동구,11250720,"MULTIPOLYGON (((127.14447 37.55477, 127.14319 ...",11740570,서울특별시,강동구,암사1동,1,주거중심지역
422,서울특별시 강동구 천호2동,1174061000,11740,11,서울특별시,강동구,11250730,"MULTIPOLYGON (((127.13458 37.54712, 127.13115 ...",11740610,서울특별시,강동구,천호2동,0,주간활성화지역
423,서울특별시 강동구 길동,1174068500,11740,11,서울특별시,강동구,11250740,"MULTIPOLYGON (((127.14857 37.54578, 127.14871 ...",11740685,서울특별시,강동구,길동,2,야간활성화지역
424,서울특별시 구로구 오류2동,1153078000,11530,11,서울특별시,구로구,11170730,"MULTIPOLYGON (((126.83179 37.47757, 126.83235 ...",11530780,서울특별시,구로구,오류2동,1,주거중심지역


In [25]:
m = folium.Map(location=[37.5665, 126.9780], zoom_start=11, tiles='OpenStreetMap')

def style_function(feature):
    cluster_nm = feature['properties']['CLUSTER_NM']
    return {
        'fillColor': CLUSTER_COLORS.get(cluster_nm, 'gray'),
        'color': 'black',
        'weight': 0.5,
        'fillOpacity': 0.7,
    }

folium.GeoJson(
    gdf_cluster,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['adm_nm', 'CLUSTER_NM'], aliases=['행정동', '유형']),
).add_to(m)

# 범례
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000; background-color: white;
            padding: 10px; border: 1px solid grey; border-radius: 5px; font-size: 14px;">
  <b>유형</b><br>
  {}
</div>
'''.format(''.join(
    f'<i style="background:{color};width:12px;height:12px;display:inline-block;margin-right:5px;"></i>{name}<br>'
    for name, color in CLUSTER_COLORS.items()
))
m.get_root().html.add_child(folium.Element(legend_html))

m.save('html/1_7.html')
m

## 7. 코사인 유사도 기반 유사 패턴 행정동 탐색

행정동별 시간대 인구 패턴의 유사성을 비교하기 위해 코사인 유사도(Cosine Similarity)를 사용한다. 이때 절대적인 인구 규모의 차이를 제거하고, 하루 동안 인구가 변화하는 **곡선의 형태**에 집중하기 위해 시간대별 인구 데이터를 z-score로 표준화한 `pattern_scaled`를 활용한다.

코사인 유사도는 두 벡터가 이루는 각도의 코사인값으로 계산한다.

$$\text{Cosine Similarity}(A, B) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

값의 범위는 -1에서 1이며, 값이 1에 가까울수록 두 행정동의 시간대별 인구 변화 패턴이 유사하다는 뜻이다.

- `1`: 시간대별 증감 흐름이 거의 동일함
- `0`: 두 패턴 사이에 뚜렷한 관계가 없음
- `-1`: 한 지역의 인구가 증가할 때 다른 지역은 감소하는 반대 패턴

각 행정동의 시간대별 데이터에 z-score 표준화를 적용하면 평균은 0, 표준편차는 1이 된다. 따라서 인구 규모가 큰 행정동과 작은 행정동도 동일한 기준에서 비교할 수 있으며, 코사인 유사도는 원래 시간대별 패턴 간 **피어슨 상관계수와 동일한 의미**를 갖는다.

이를 통해 주거 중심, 업무 중심, 상업·관광 중심 등 하루 인구 변화 양상이 비슷한 행정동을 탐색할 수 있다.

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

target_dong = '역삼1동'
target_cd = dong_cluster.loc[dong_cluster['ADMI_NM'] == target_dong, 'H_DNG_CD'].iloc[0]

X = pattern_scaled[hour_cols]
sim = cosine_similarity(X.loc[[target_cd]], X)[0]
similarity = pd.Series(sim, index=X.index, name='COS_SIM')

dong_similarity = (
    dong_cluster[['H_DNG_CD', 'SGG_NM', 'ADMI_NM', 'CLUSTER_NM']]
    .drop_duplicates('H_DNG_CD')
    .merge(similarity, left_on='H_DNG_CD', right_index=True)
    .sort_values('COS_SIM', ascending=False)
    .reset_index(drop=True)
)

# 자기 자신을 제외하고 가장 유사한 행정동 순
dong_similarity[dong_similarity['H_DNG_CD'] != target_cd].head(10)

,H_DNG_CD,SGG_NM,ADMI_NM,CLUSTER_NM,COS_SIM
1,11680521,강남구,논현1동,주간활성화지역,0.998957
2,11110615,종로구,종로1.2.3.4가동,주간활성화지역,0.997839
3,11140570,중구,필동,주간활성화지역,0.997300
4,11440600,마포구,대흥동,주간활성화지역,0.996913
5,11200660,성동구,성수1가2동,주간활성화지역,0.996368
6,11140605,중구,을지로동,주간활성화지역,0.996069
7,11410585,서대문구,신촌동,주간활성화지역,0.996046
8,11305595,강북구,번1동,주간활성화지역,0.995671
9,11560605,영등포구,문래동,주간활성화지역,0.995303
10,11500535,강서구,등촌3동,주간활성화지역,0.995025


In [27]:
# 역삼1동 + 가장 유사한 5개 행정동의 시간대별 패턴(표준화 값) 시각화
top5 = dong_similarity[dong_similarity['H_DNG_CD'] != target_cd].head(5)
target_and_top5 = pd.concat([dong_similarity[dong_similarity['H_DNG_CD'] == target_cd], top5])

plot_df = (
    pattern_scaled.loc[target_and_top5['H_DNG_CD'], hour_cols]
    .reset_index()
    .merge(target_and_top5[['H_DNG_CD', 'ADMI_NM', 'COS_SIM']], on='H_DNG_CD')
)

plot_long = plot_df.melt(id_vars=['H_DNG_CD', 'ADMI_NM', 'COS_SIM'], var_name='HOUR', value_name='Z_SPOP')
plot_long['HOUR'] = plot_long['HOUR'].str.replace('H', '').astype(int)

fig = px.line(
    plot_long,
    x='HOUR', y='Z_SPOP', color='ADMI_NM',
    markers=True,
    labels={'HOUR': 'Hour', 'Z_SPOP': '표준화된 생활인구', 'ADMI_NM': '행정동'},
    title=f'{target_dong}과 가장 유사한 5개 행정동의 시간대별 생활인구 패턴',
    template='simple_white',
)
fig.update_xaxes(dtick=1)
fig.update_traces(selector={'name': target_dong}, line={'width': 4, 'dash': 'dash'})

fig.show()
fig.write_html("html/1_8.html")
